In [7]:
!unzip src.zip

Archive:  src.zip
   creating: src/api/
   creating: src/data/
  inflating: src/data/data_loader.py  
  inflating: src/data/feature_engineering.py  
  inflating: src/data/preprocessing.py  
  inflating: src/data/__init__.py    
   creating: src/data/__pycache__/
  inflating: src/data/__pycache__/data_loader.cpython-311.pyc  
  inflating: src/data/__pycache__/feature_engineering.cpython-311.pyc  
  inflating: src/data/__pycache__/feature_engineering.cpython-313.pyc  
  inflating: src/data/__pycache__/preprocessing.cpython-311.pyc  
  inflating: src/data/__pycache__/preprocessing.cpython-313.pyc  
  inflating: src/data/__pycache__/__init__.cpython-311.pyc  
  inflating: src/data/__pycache__/__init__.cpython-313.pyc  
   creating: src/evaluation/
  inflating: src/evaluation/evaluator.py  
  inflating: src/evaluation/visualization.py  
  inflating: src/evaluation/__init__.py  
   creating: src/evaluation/__pycache__/
  inflating: src/evaluation/__pycache__/evaluator.cpython-311.pyc  
  inf

In [8]:
!unzip datasets.zip

Archive:  datasets.zip
   creating: datasets/
   creating: datasets/bed_availability/
   creating: datasets/processed/
  inflating: datasets/processed/bed_availability_processed.csv  
  inflating: datasets/processed/eta_processed.csv  
  inflating: datasets/processed/severity_cleaned.csv  
  inflating: datasets/processed/severity_processed.csv  
   creating: datasets/raw/
  inflating: datasets/raw/5v_cleandf.rdata  
  inflating: datasets/raw/converted_data.csv  
  inflating: datasets/raw/nyc_taxi_data.csv  
  inflating: datasets/raw/README_data_sources.md  
  inflating: datasets/raw/weather_nyc_2023.csv  
   creating: datasets/synthetic/


In [9]:
!pip install xgboost lightgbm scikit-learn pandas numpy

In [10]:
import sys
sys.path.append('/content')
sys.path.append('/content/src')

In [11]:
import os

def optimize_file(filepath):
    if not os.path.exists(filepath):
        print(f'File {filepath} not found, skipping.')
        return

    with open(filepath, 'r') as f:
        content = f.read()

    # Optimization 1 & 2: n_jobs, n_estimators, max_depth
    content = content.replace('n_jobs=1', 'n_jobs=-1')
    content = content.replace('n_estimators=100', 'n_estimators=50')
    content = content.replace('max_depth=6', 'max_depth=3')

    # Optimization 3: Remove slow regressors if they are being imported/used
    content = content.replace('from sklearn.ensemble import GradientBoostingRegressor', '# Removed GradientBoosting')
    content = content.replace('from sklearn.ensemble import StackingRegressor', '# Removed Stacking')

    with open(filepath, 'w') as f:
        f.write(content)
    print(f'Optimized {filepath}')

# Target files based on your directory structure
targets = [
    'src/models/eta_predictor.py',
    'src/models/bed_predictor.py',
    'src/models/severity_classifier.py',
    'src/models/model_trainer.py'
]

for t in targets:
    optimize_file(t)

Optimized src/models/eta_predictor.py
Optimized src/models/bed_predictor.py
Optimized src/models/severity_classifier.py
Optimized src/models/model_trainer.py


In [12]:
!python train_all_models.py

Starting master training script...
2026-04-02 19:24:57 - master_training - INFO - Starting Full Model Training Pipeline...
2026-04-02 19:24:57 - eta_trainer - INFO - ============================================================
2026-04-02 19:24:57 - eta_trainer - INFO - TRAINING ETA PREDICTION MODELS
2026-04-02 19:24:57 - eta_trainer - INFO - ============================================================
2026-04-02 19:24:59 - eta_trainer - INFO - Training Random Forest...

Model: Random Forest
MAE: 3.6770 | R²: 0.7324
✅ Registered eta_random_forest v1
2026-04-02 19:32:20 - eta_trainer - INFO - Training XGBoost...

Model: XGBoost
MAE: 3.4802 | R²: 0.7521
✅ Registered eta_xgboost v1
2026-04-02 19:32:29 - eta_trainer - INFO - Training LightGBM...
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(

Model: LightGBM
MAE: 3.4899 | R²: 0.7511
✅ Registered eta_l

In [23]:
with open('src/data/data_loader.py', 'r') as f:
    print(f.read())

import pandas as pd
import os
from ..utils.config import Config

def load_severity_data():
    """Load severity dataset (from the real processed real data)."""
    path = Config.SEVERITY_CLEANED_CSV
    if not os.path.exists(path):
        raise FileNotFoundError(f"Real data not found at {path}.")
    return pd.read_csv(path)

def load_eta_data():
    """Load ETA dataset."""
    path = Config.ETA_PROCESSED_CSV
    if not os.path.exists(path):
        raise FileNotFoundError(f"ETA data not found at {path}.")
    return pd.read_csv(path)

def load_bed_data():
    """Load hospital beds dataset (from your originally processed bed data)."""
    path = Config.BED_PROCESSED_CSV
    if not os.path.exists(path):
        raise FileNotFoundError(f"Bed data not found at {path}.")
    return pd.read_csv(path)



In [19]:
with open('src/utils/model_registry.py', 'r') as f:
    print(f.read())

"""
Model Registry — tracks model versions with metrics and metadata.
Stores versioning info in a JSON file for easy retrieval.
"""

import json
import os
import joblib
from datetime import datetime


class ModelRegistry:
    """JSON-based model version tracking."""
    
    def __init__(self, registry_path="models/model_registry.json"):
        self.registry_path = registry_path
        self.registry = self._load_registry()
    
    def _load_registry(self):
        """Load existing registry or create new one."""
        try:
            with open(self.registry_path, "r") as f:
                data = json.load(f)
                if "models" not in data:
                    data["models"] = []
                return data
        except (FileNotFoundError, json.JSONDecodeError):
            return {"models": []}
    
    def register_model(self, model, model_name, metrics, version=None, save=True):
        """
        Register a trained model with metrics.
        
        Args:
       

In [20]:
with open('src/utils/model_registry.py', 'r') as f:
    print(f.read())

"""
Model Registry — tracks model versions with metrics and metadata.
Stores versioning info in a JSON file for easy retrieval.
"""

import json
import os
import joblib
from datetime import datetime


class ModelRegistry:
    """JSON-based model version tracking."""
    
    def __init__(self, registry_path="models/model_registry.json"):
        self.registry_path = registry_path
        self.registry = self._load_registry()
    
    def _load_registry(self):
        """Load existing registry or create new one."""
        try:
            with open(self.registry_path, "r") as f:
                data = json.load(f)
                if "models" not in data:
                    data["models"] = []
                return data
        except (FileNotFoundError, json.JSONDecodeError):
            return {"models": []}
    
    def register_model(self, model, model_name, metrics, version=None, save=True):
        """
        Register a trained model with metrics.
        
        Args:
       

In [26]:
import os

eta_predictor_path = 'src/models/eta_predictor.py'

# Rewriting the file to use only XGBoost and LightGBM with correct registry usage
new_content = """
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from src.utils.model_registry import ModelRegistry
from src.data.data_loader import load_eta_data

def train_eta_models():
    df = load_eta_data()

    # Basic preprocessing: assuming 'target' is the column to predict
    target_col = 'trip_duration' if 'trip_duration' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    registry = ModelRegistry()

    # Using only XGBoost and LightGBM for speed and power
    models = {
        'XGBoost': XGBRegressor(n_estimators=50, max_depth=3, n_jobs=-1),
        'LightGBM': LGBMRegressor(n_estimators=50, max_depth=3, n_jobs=-1)
    }

    trained_models = {}
    for name, model in models.items():
        print(f'Training {name}...')
        model.fit(X_train, y_train)

        # Evaluate simple metrics for registration
        y_pred = model.predict(X_test)
        mae = np.mean(np.abs(y_test - y_pred))

        # Register model using the registry instance
        registry.register_model(
            model=model,
            model_name=f"eta_{name.lower()}",
            metrics={'mae': mae}
        )
        trained_models[name] = model

    return trained_models
"""

with open(eta_predictor_path, 'w') as f:
    f.write(new_content.strip())

print(f'Successfully updated {eta_predictor_path} with XGBoost and LightGBM.')

Successfully updated src/models/eta_predictor.py with XGBoost and LightGBM.


In [27]:
!python train_all_models.py

Starting master training script...
2026-04-02 20:13:17 - master_training - INFO - Starting Full Model Training Pipeline...
Training XGBoost...
✅ Registered eta_xgboost v3
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027140 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1320
[LightGBM] [Info] Number of data points in the train set: 1158204, number of used features: 9
[LightGBM] [Info] Start training from score 14.035083
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [28]:
!python train_all_models.py

Starting master training script...
2026-04-02 20:15:17 - master_training - INFO - Starting Full Model Training Pipeline...
Training XGBoost...
✅ Registered eta_xgboost v4
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027211 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1320
[LightGBM] [Info] Number of data points in the train set: 1158204, number of used features: 9
[LightGBM] [Info] Start training from score 14.035083
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [29]:
import os

bed_predictor_path = 'src/models/bed_predictor.py'

# Optimized version of bed_predictor.py without the invalid 'use_rich' argument
new_bed_content = """
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from src.utils.model_registry import ModelRegistry
from src.data.data_loader import load_bed_data

def train_bed_models():
    # Removed the incorrect use_rich=False argument
    df = load_bed_data()

    # Basic preprocessing: using 'available_beds' or similar as target
    target_col = 'available_beds' if 'available_beds' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    registry = ModelRegistry()

    models = {
        'XGBoost': XGBRegressor(n_estimators=50, max_depth=3, n_jobs=-1),
        'LightGBM': LGBMRegressor(n_estimators=50, max_depth=3, n_jobs=-1)
    }

    trained_models = {}
    for name, model in models.items():
        print(f'Training Bed Model: {name}...')
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mae = np.mean(np.abs(y_test - y_pred))

        registry.register_model(
            model=model,
            model_name=f"bed_{name.lower()}",
            metrics={'mae': mae}
        )
        trained_models[name] = model

    return trained_models
"""

with open(bed_predictor_path, 'w') as f:
    f.write(new_bed_content.strip())

print(f'Successfully updated {bed_predictor_path} to fix the TypeError.')

Successfully updated src/models/bed_predictor.py to fix the TypeError.


In [30]:
!python train_all_models.py

Starting master training script...
2026-04-02 20:18:33 - master_training - INFO - Starting Full Model Training Pipeline...
Training XGBoost...
✅ Registered eta_xgboost v5
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.142828 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1320
[LightGBM] [Info] Number of data points in the train set: 1158204, number of used features: 9
[LightGBM] [Info] Start training from score 14.035083
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

In [31]:
!python train_all_models.py

Starting master training script...
2026-04-02 20:30:03 - master_training - INFO - Starting Full Model Training Pipeline...
Training XGBoost...
✅ Registered eta_xgboost v6
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066806 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1320
[LightGBM] [Info] Number of data points in the train set: 1158204, number of used features: 9
[LightGBM] [Info] Start training from score 14.035083
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [33]:
import joblib
from src.utils.model_registry import ModelRegistry

# Initialize the registry to find trained models
registry = ModelRegistry()

try:
    # Attempt to load the active XGBoost ETA model from the registry
    model = registry.get_active_model('eta_xgboost')
    joblib.dump(model, "model.pkl")
    print("Successfully loaded 'eta_xgboost' from registry and saved to model.pkl")
except Exception as e:
    print(f"Could not load model: {e}")

Successfully loaded 'eta_xgboost' from registry and saved to model.pkl


In [35]:
!python train_all_models.py

Starting master training script...
2026-04-02 20:51:51 - master_training - INFO - Starting Full Model Training Pipeline...
Training XGBoost...
✅ Registered eta_xgboost v7
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027491 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1320
[LightGBM] [Info] Number of data points in the train set: 1158204, number of used features: 9
[LightGBM] [Info] Start training from score 14.035083
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [37]:
import joblib
from src.utils.model_registry import ModelRegistry

# Initialize the registry to access saved models
registry = ModelRegistry()

try:
    # Load the active versions from the registry
    bed_xgb = registry.get_active_model('bed_xgboost')
    bed_lgbm = registry.get_active_model('bed_lightgbm')

    # Save them to the desired .pkl files
    joblib.dump(bed_xgb, 'bed_xgboost.pkl')
    joblib.dump(bed_lgbm, 'bed_lightgbm.pkl')
    print("Successfully saved 'bed_xgboost.pkl' and 'bed_lightgbm.pkl'")
except Exception as e:
    print(f"Could not load models: {e}. Ensure the training script has completed for bed models.")

Successfully saved 'bed_xgboost.pkl' and 'bed_lightgbm.pkl'


In [39]:
import joblib
# Replace 'my_model' with the actual name you want for the file
joblib.dump(model, 'my_model.pkl')

['my_model.pkl']

In [40]:
!ls

bed_lightgbm.pkl  datasets.zip	models	      sample_data  train_all_models.py
bed_xgboost.pkl   logs		my_model.pkl  src
datasets	  model.pkl	results       src.zip


In [43]:
from google.colab import files
files.download('bed_lightgbm.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>